# Proyecto 3 · Notebook completo (Colab T4)

Flujo completo en **una sola sesión**: datos → entrenamiento LoRA (configs A y B) → evaluación → entregables.
Cada bloque tiene un "guard" (`ya está → skip`) y un **checkpoint de git** (commit + push),
así que si la sesión de Colab se cae, reabras este mismo notebook y sigue donde quedó.

Antes de empezar:
1. En el panel izquierdo, **ícono de llave (Secretos)**: agrega `GITHUB_TOKEN` (un token fine-grained
   de GitHub con permiso *Contents: Read and write*) y actívelo con "acceso para este notebook".
2. Llena `REPO`, `GIT_USER` y `GIT_EMAIL` en la celda siguiente.
3. **Entorno de ejecución → Cambiar tipo → T4 GPU**.


In [ ]:
# --- 0 · Setup: clona el repo, instala dependencias ---
import os, subprocess
REPO = "USUARIO/proyecto3"          # <-- CAMBIAR: usuario/repo
BRANCH = "main"
RUTA = "/content/proyecto3"

GIT_USER = "José Ordóñez"           # <-- CAMBIAR: tu nombre (los checkpoint deben salir de tu cuenta)
GIT_EMAIL = "jose@..."              # <-- CAMBIAR: tu email de GitHub

if not os.path.exists(RUTA):
    from google.colab import userdata
    TOKEN = userdata.get("GITHUB_TOKEN")
    subprocess.run(f"git clone -b {BRANCH} https://{TOKEN}@github.com/{REPO}.git {RUTA}",
                   shell=True, check=True)
os.chdir(RUTA)
subprocess.run(["git", "config", "user.name", GIT_USER])
subprocess.run(["git", "config", "user.email", GIT_EMAIL])
print("repo:", subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout.strip())


In [ ]:
# (solo la primera vez: instala dependencias; luego ya no se necesita re-instalar)
!pip install -q -r requirements.txt 2>&1 | tail -1
# peft 0.21 exige torchao>=0.16 y Colab trae 0.10; no lo usamos (es para cuantización), lo sacamos
!pip uninstall -q -y torchao 2>&1 | tail -1
import os
assert os.path.exists("solucion.py"), "no estás en la raíz del repo (re-corre la celda de setup)"

def checkpoint(mensaje):
    """commit + push de todo lo producido hasta ahora (reanudable)."""
    import subprocess
    subprocess.run(["git", "config", "user.name", GIT_USER])
    subprocess.run(["git", "config", "user.email", GIT_EMAIL])
    subprocess.run(["git", "add", "-A"], capture_output=True)
    r = subprocess.run(["git", "commit", "-m", mensaje], capture_output=True, text=True)
    if "nothing to commit" in (r.stdout + r.stderr):
        print("(checkpoint: nada nuevo que commitear)")
        return
    subprocess.run(["git", "pull", "--rebase", "origin", BRANCH], capture_output=True, text=True)
    p = subprocess.run(["git", "push", "origin", "HEAD"], capture_output=True, text=True)
    if p.returncode != 0:
        print("push FALLÓ:\n", p.stderr)
    else:
        print(f"push ok: '{mensaje}'")

def correr(comando):
    """Corre un comando mostrando su salida EN VIVO y DETIENE la celda si falla.

    Sustituye a os.system, cuya salida Colab a veces traga y que no corta
    la celda con el error real (el FileNotFound engañoso que ya vimos)."""
    import subprocess, time
    t0 = time.time()
    p = subprocess.Popen(comando, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    p.wait()
    if p.returncode != 0:
        raise RuntimeError(f"\n[error] '{comando}' falló con código {p.returncode}")
    print(f"[ok] {round(time.time() - t0)}s")


In [ ]:
# Verificar GPU (runtime T4)
!nvidia-smi | head -12


## 1 · Datos: MySQL 8.4.10 y esquemas

Los dumps y los 2,840 pares ya están en el repo; aquí solo se cargan en MySQL (y se reconstruyen
las `SHOW CREATE TABLE`) porque el evaluador necesita una base viva.


In [ ]:
# --- 1a · MySQL (idempotente) ---
!bash scripts/instalar_mysql_colab.sh | tail -2
!python datos/construir_esquemas.py --socket /tmp/mysql.sock --crear-lector | tail -4


In [ ]:
# --- 1b · dataset en formato de chat (con guard) ---
import os
def hecho(ruta): return os.path.exists(ruta)
if not hecho("datos/entrenamiento/dataset_chat.jsonl"):
    correr("python src/datos_chat.py")
else:
    print("dataset_chat.jsonl ya existe (skip)")
checkpoint("datos listos en Colab (esquemas cargados en MySQL 8.4.10)")


## 2 · Entrenamiento LoRA

Configs A (atención) y B (atención + MLP) — `r=16, alpha=32, lr=2e-4, 1 época, batch efectivo 16,
semilla 2026`; el factor único es `target_modules` (hipótesis en `resultados/hipotesis.md`).
Cada una toma 35–50 min en T4.


In [ ]:
# --- 2a · Config A (atención) ---
import os
if not os.path.exists("adapters/config_A/adapter_config.json"):
    correr("python src/entrenar.py --config A")
else:
    print("config A ya entrenada (skip)")
import json
rA = json.load(open("resultados/entrenamiento_A.json"))
print("A · entrenables: {:,} ({} %) · min: {} · eval_loss final: {:.4f}".format(
      rA["parametros_entrenables"], round(rA["porcentaje_entrenable"], 3),
      rA["minutos_entrenamiento"], rA["eval_loss_final"]))


In [ ]:
# --- 2b · Config B (atención + MLP) ---
import os
if not os.path.exists("adapters/config_B/adapter_config.json"):
    correr("python src/entrenar.py --config B")
else:
    print("config B ya entrenada (skip)")
import json
rB = json.load(open("resultados/entrenamiento_B.json"))
print("B · entrenables: {:,} ({} %) · min: {} · eval_loss final: {:.4f}".format(
      rB["parametros_entrenables"], round(rB["porcentaje_entrenable"], 3),
      rB["minutos_entrenamiento"], rB["eval_loss_final"]))


In [ ]:
checkpoint("Entrenamiento LoRA de las configs A (atención) y B (atención+MLP)")


### Curvas de loss


In [ ]:
from IPython.display import Image
Image("resultados/loss_A.png")


In [ ]:
Image("resultados/loss_B.png")


## 3 · Evaluación (exactitud por ejecución, igual criterio que el docente)

Baseline (sin adapter) y las dos configs con greedy; luego experimentos de inferencia:
beam 4, esquema crudo, y `ejecutar` + reparar.


In [ ]:
# --- 3a · Baseline y configs (skip las ya evaluadas) ---
import os
variantes = [
    ("base_greedy", "--sin-adapter --beams 1"),
    ("A_greedy",    "--adapter adapters/config_A --beams 1"),
    ("B_greedy",    "--adapter adapters/config_B --beams 1"),
]
for nombre, flags in variantes:
    if os.path.exists(f"resultados/eval_{nombre}.json"):
        print(f"{nombre}: ya evaluada (skip)")
        continue
    print(f"=== evaluando {nombre} ===")
    correr(f"python src/correr_evaluacion.py --nombre {nombre} {flags} --socket /tmp/mysql.sock")


In [ ]:
# --- 3b · Elegir el adapter ganador (compara con la hipótesis) ---
import json
base = json.load(open("resultados/eval_base_greedy.json"))["EX_ponderada"]
A = json.load(open("resultados/eval_A_greedy.json"))["EX_ponderada"]
B = json.load(open("resultados/eval_B_greedy.json"))["EX_ponderada"]
MEJOR = "A" if A >= B else "B"
ADAPTER = f"adapters/config_{MEJOR}"
print(f"base={base:.3f} · A={A:.3f} · B={B:.3f}")
print(f"mejora del mejor adapter sobre el baseline: {max(A, B) - base:+.3f}")
print("ganador: config", MEJOR, "→", ADAPTER)


In [ ]:
# --- 3c · Experimentos de inferencia con el adapter ganador ---
import os
variantes_exp = [
    ("mejor_beam",  f"--adapter {ADAPTER} --beams 4"),
    ("mejor_crudo", f"--adapter {ADAPTER} --beams 1 --formato crudo"),
    ("base_final",  "--sin-adapter --beams 4"),
    ("final",       f"--adapter {ADAPTER} --beams 4"),
]
for nombre, flags in variantes_exp:
    if os.path.exists(f"resultados/eval_{nombre}.json"):
        print(f"{nombre}: ya evaluada (skip)")
        continue
    print(f"=== evaluando {nombre} ===")
    correr(f"python src/correr_evaluacion.py --nombre {nombre} {flags} --socket /tmp/mysql.sock")
checkpoint("Evaluaciones de validación propia (baseline, configs A/B y experimentos de inferencia)")


### Resumen del experimento de inferencia


In [ ]:
import json
print(f"{'corrida':>14}   EX     · por nivel (F/M/D)")
for n in ["base_greedy", "A_greedy", "B_greedy", "mejor_beam", "mejor_crudo", "base_final", "final"]:
    e = json.load(open(f"resultados/eval_{n}.json"))
    pn = e["por_nivel"]
    print(f"{n:>14}   {e['EX_ponderada']:.3f}  ({pn['facil']:.2f} / {pn['media']:.2f} / {pn['dificil']:.2f})")


## 4 · Entregables: adapter final, predicción, presentación e historial git


In [ ]:
# --- 4a · Copiar el adapter ganador a adapter/ (lo que ejecuta el docente) ---
import shutil, subprocess
shutil.rmtree("adapter", ignore_errors=True)   # borra LEEME.md y restos
shutil.copytree(ADAPTER, "adapter")
print(subprocess.run(["ls", "-la", "adapter/"], capture_output=True, text=True).stdout)


In [ ]:
# --- 4b · Predicción justificada del EX oculto (se basa en el CSV de la corrida "final") ---
!python src/prediccion.py


In [ ]:
# --- 4c · Presentación (12 diapositivas) ---
!python src/generar_presentacion.py --autores "Adrián González" "José Ordóñez"


In [ ]:
# --- 4d · Verificación de contrato, como la hará el docente ---
#   carga < 5 min, < 30 s por pregunta, determinismo, solo lectura
import time, json, sys
sys.path.insert(0, ".")
t0 = time.time()
import solucion
print("config.json:", json.load(open("config.json")))
print("carga completa:", round(time.time() - t0, 1), "s (máx. 5 min)")
esquema = open("datos/esquemas/val/biblioteca.schema.sql").read()
p = json.load(open("datos/validacion/validacion_propia.json"))[2]
t0 = time.time()
s1 = solucion.generar_sql(esquema, p["pregunta"])
t1 = time.time()
s2 = solucion.generar_sql(esquema, p["pregunta"])
print("pregunta:", p["pregunta"])
print("SQL:", s1[:140])
print("determinista:", s1 == s2, "·", round(t1 - t0, 2), "s la primera y", round(time.time() - t1, 2), "s la segunda")
assert s1 == s2 and solucion.es_segura(s1)
checkpoint("Adapter final en adapter/, validacion_propia.csv, prediccion.json y presentacion.pdf")


In [ ]:
# --- 4e · historial_git.txt (traduce el log completo con lo de esta sesión incluida) ---
!git log --all --date=iso --stat > historial_git.txt
!head -30 historial_git.txt
checkpoint("historial_git.txt")


## 5 · Resumen final


In [ ]:
import json, os
print("checklist de entregables del PDF:")
for ruta in ["solucion.py", "config.json", "requirements.txt", "adapter/adapter_config.json",
             "datos/entrenamiento/train.jsonl", "datos/entrenamiento/dataset_chat.jsonl",
             "datos/validacion/validacion_propia.json", "datos/descartes/resumen_descartes.json",
             "resultados/validacion_propia.csv", "resultados/prediccion.json",
             "presentacion.pdf", "historial_git.txt", "BITACORA.md", "CONTRIBUCIONES.md"]:
    print("  [OK]   " + ruta if os.path.exists(ruta) else "  [FALTA] " + ruta)

print("\nEX ponderada (EX) por variante:")
final = json.load(open("resultados/eval_final.json"))["EX_ponderada"]
base_fin = json.load(open("resultados/eval_base_final.json"))["EX_ponderada"]
mejora = final - base_fin
gravar = {"base_greedy": "baseline (sin adapter)", "A_greedy": "LoRA A", "B_greedy": "LoRA B",
          "mejor_beam": "mejor config + beam 4", "mejor_crudo": "mejor config + esquema crudo",
          "base_final": "base + beam + ejecutar", "final": "ENTREGA (adapter + beam + ejecutar)"}
for n, desc in gravar.items():
    e = json.load(open(f"resultados/eval_{n}.json"))
    print(f"  {n:>13} · {desc:<32} · EX {e['EX_ponderada']:.3f}")
print(f"\nMejora adapter vs base con el MISMO pipeline: {mejora:+.3f} (el PDF pide >= +0.15 para los 10 pts)")
print(f"EX de la entrega: {final:.3f} (el PDF da los 30 pts si >= 0.70)")
print("\npredicción para el examen oculto:")
print(open("resultados/prediccion.json").read()[:700])
